# Handwired — GPU Contact Hardware Verification
Uses Kaggle's Tesla T4 GPUs, installs Chrome, and conditionally installs matching NVIDIA Vulkan userspace ICD to test real Chromium WebGPU. Rejects software fallback. CUDA detection alone is not GPU browser verification.


In [ ]:
import os, subprocess, shutil, pathlib, json, traceback, time
workspace=pathlib.Path('/kaggle/working')
report={'started':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'hardwareVerified':False,'steps':[]}
def run(label,cmd,check=True,env=None):
    print('---',label,'---',flush=True)
    try:
        proc=subprocess.run(cmd,shell=True,executable='/bin/bash',cwd='/kaggle/working',env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=1600)
        print(proc.stdout[-16000:],flush=True)
        item={'step':label,'returncode':proc.returncode,'outputTail':proc.stdout[-4000:]}
        report['steps'].append(item)
        if check and proc.returncode!=0: raise RuntimeError(f'{label} failed ({proc.returncode})')
        return proc
    finally:
        (workspace/'handwired-kaggle-report.json').write_text(json.dumps(report,indent=2))
try:
    run('GPU identity','nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader')
    run('Install Chrome and Vulkan metadata',"""apt-get update -qq && apt-get install -y -qq wget libvulkan1 vulkan-tools libnss3 libgbm1 && wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/google-chrome.deb && apt-get install -y -qq /tmp/google-chrome.deb""")
    run('NVIDIA Vulkan ICD for driver series 580','apt-get install -y -qq libnvidia-gl-580',check=False)
    run('Chrome and Vulkan diagnostics','google-chrome --version; vulkaninfo --summary | head -90',check=False)
    run('Install Bun', 'curl -fsSL https://bun.sh/install | bash')
    run('Fetch public Handwired branch','git clone --depth 1 --branch feat/frontiers-gpu-contacts https://github.com/Chrisbryan17/handwired.git /kaggle/temp/handwired')
    run('Bun pinned dependency install','cd /kaggle/temp/handwired && /root/.bun/bin/bun install --frozen-lockfile')
    env=os.environ.copy()
    env['HANDWIRED_GPU_BROWSER']='/usr/bin/google-chrome'
    env['HANDWIRED_BROWSER_FLAGS']='--use-angle=vulkan --enable-features=Vulkan'
    run('CPU oracle and TypeScript','cd /kaggle/temp/handwired && /root/.bun/bin/bun test lab/frontiers/destruction/model.test.ts src/runtime/frontiers-contacts.test.ts && /root/.bun/bin/bunx tsc -p lab/frontiers/destruction/tsconfig.json --pretty false',env=env)
    run('Hardware GPU contact parity','cd /kaggle/temp/handwired && /root/.bun/bin/bun scripts/verify-contacts-gpu-browser.ts /kaggle/working/hardware-contacts.json --screenshot',env=env)
    run('Hardware 3D contact scene','cd /kaggle/temp/handwired && /root/.bun/bin/bun scripts/verify-destruction-demo-browser.ts /kaggle/working/hardware-impact.json --steps=72 --screenshot',env=env)
    reports=[]
    for file in ['hardware-contacts.json','hardware-impact.json']:
        p=workspace/file
        if p.exists(): reports.append(json.loads(p.read_text()))
    report['hardwareVerified']=len(reports)==2 and all(r.get('ok') is True and r.get('hardware') is True for r in reports)
    report['testReports']=reports
except Exception as exc:
    report['error']=str(exc)
    traceback.print_exc()
finally:
    report['finished']=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())
    (workspace/'handwired-kaggle-report.json').write_text(json.dumps(report,indent=2))
print('FINAL HARDWARE VERIFIED',report['hardwareVerified'],flush=True)
